In [1]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from IPython.display import Markdown
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

basic_llm = ChatOllama(model="gemma3:1b", temperature=0)
advenced_llm = ChatOllama(model="llama3.2:1b", temperature=0)

@wrap_model_call
def dynamic_model_selection(request:ModelRequest, handler)->ModelResponse:
    env = request.runtime.context.get("env","test")
    if env=="prod":
        model = advenced_llm
        print("advenced_llm selected ...")
    else:
        model = basic_llm
        print("basic_llm selected ...")
    return handler(request.override(model=model))

configure = {"configurable":{"thread_id":11}}
memory = InMemorySaver()
agent2 = create_agent(
    model=basic_llm,
    tools=[],
    middleware=[dynamic_model_selection],
    # debug=True,
    checkpointer=memory
)
directive="reponse courte et ne me pose pas de question"

msg1 = "le maroc est un beau pays ?"
print(msg1)
resp = agent2.invoke(
    input={"messages":[HumanMessage(msg1 + directive )]},
    config=configure,
    context={"env":"prod"}
)
# print(display(Markdown(resp["messages"][-1].content)))
print(resp["messages"][-1].content)


msg2 = "comment s'appelle sa capitale ? "
print(msg2)
resp2 = agent2.invoke(
    input={"messages":[HumanMessage( msg2 + directive)]},
    config=configure,
    context={"env":"prod"}
    )
# print(display(Markdown(resp2["messages"][-1].content)))
print(resp2["messages"][-1].content)

e:\Projets\AgenticIA\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


le maroc est un beau pays ?
advenced_llm selected ...
Oui, le Maroc est un pays très riche en culture, histoire et nature. Ses plages de sable blanc, ses montagnes de roche rouge, ses vallées verdoyantes...
comment s'appelle sa capitale ? 
advenced_llm selected ...
Rabat.
